# 电商销售数据分析

## 分析目标
- 理解销售整体状况
- 识别高价值用户
- 发现畅销品类
- 分析物流时效

---

In [ ]:
# 导入库
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# 设置中文显示
plt.rcParams['font.sans-serif'] = ['SimHei', 'DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False

print("库导入成功！")

In [ ]:
# 加载数据
df = pd.read_csv('data/ecommerce_data.csv')

# 查看数据基本信息
print(f"数据集形状: {df.shape}")
print(f"\n列信息:")
df.info()

# 查看前5行
df.head()

In [ ]:
# 数据清洗：检查缺失值
print("缺失值统计:")
print(df.isnull().sum())

# 数据清洗：检查异常值
print("\n数值列统计描述:")
df.describe()

In [ ]:
# 特征工程：计算订单总金额
df['total_amount'] = df['quantity'] * df['unit_price'] * (1 - df['discount']) + df['shipping_cost']

# 转换日期格式
df['order_date'] = pd.to_datetime(df['order_date'])
df['month'] = df['order_date'].dt.month
df['day_of_week'] = df['order_date'].dt.dayofweek

print("特征工程完成！")
print(f"总销售额: ${df['total_amount'].sum():,.2f}")
print(f"平均订单金额: ${df['total_amount'].mean():,.2f}")

In [ ]:
# 分析1：各品类销售额分布
category_sales = df.groupby('product_category')['total_amount'].sum().sort_values(ascending=False)

plt.figure(figsize=(10, 6))
category_sales.plot(kind='bar', color='steelblue')
plt.title('各品类销售额分布', fontsize=14)
plt.xlabel('商品类别')
plt.ylabel('销售额($)')
plt.xticks(rotation=45)
plt.tight_layout()
plt.savefig('outputs/charts/category_sales.png', dpi=100)
plt.show()

print("畅销品类 Top 3:")
print(category_sales.head(3))

In [ ]:
# 分析2：用户价值分析（RFM模型简化版）
rfm = df.groupby('customer_id').agg({
    'order_date': lambda x: (pd.Timestamp('2024-01-31') - x.max()).days,  # Recency
    'order_id': 'count',  # Frequency
    'total_amount': 'sum'  # Monetary
}).rename(columns={'order_date': 'recency', 'order_id': 'frequency', 'total_amount': 'monetary'})

print("RFM分析结果:")
print(rfm.sort_values('monetary', ascending=False).head(10))

# 识别高价值用户
high_value = rfm[rfm['monetary'] > rfm['monetary'].quantile(0.75)]
print(f"\n高价值用户数量: {len(high_value)}")
print(f"高价值用户贡献占比: {high_value['monetary'].sum() / rfm['monetary'].sum() * 100:.1f}%")

In [ ]:
# 分析3：支付方式偏好
payment_stats = df.groupby('payment_method').agg({
    'total_amount': 'sum',
    'order_id': 'count'
}).rename(columns={'total_amount': '销售额', 'order_id': '订单数'})

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# 销售额占比
payment_stats['销售额'].plot(kind='pie', ax=axes[0], autopct='%1.1f%%', startangle=90)
axes[0].set_title('各支付方式销售额占比')
axes[0].set_ylabel('')

# 订单数
payment_stats['订单数'].plot(kind='bar', ax=axes[1], color='coral')
axes[1].set_title('各支付方式订单数')
axes[1].set_xlabel('支付方式')
axes[1].set_ylabel('订单数')

plt.tight_layout()
plt.savefig('outputs/charts/payment_analysis.png', dpi=100)
plt.show()

print(payment_stats)

In [ ]:
# 分析4：物流时效分析
delivery_stats = df.groupby('customer_segment')['delivery_days'].agg(['mean', 'min', 'max'])

plt.figure(figsize=(8, 5))
df.boxplot(column='delivery_days', by='customer_segment')
plt.title('不同用户群体配送天数分布')
plt.suptitle('')
plt.xlabel('用户类型')
plt.ylabel('配送天数')
plt.savefig('outputs/charts/delivery_analysis.png', dpi=100)
plt.show()

print("不同用户群体配送时效:")
print(delivery_stats)

In [ ]:
# 分析5：折扣策略效果
discount_analysis = df.groupby('discount').agg({
    'total_amount': 'mean',
    'order_id': 'count'
}).rename(columns={'total_amount': '平均订单金额', 'order_id': '订单数'})

print("折扣与订单金额关系:")
print(discount_analysis)

# 有折扣vs无折扣
avg_order_with_discount = df[df['discount'] > 0]['total_amount'].mean()
avg_order_without_discount = df[df['discount'] == 0]['total_amount'].mean()

print(f"\n有折扣平均订单金额: ${avg_order_with_discount:.2f}")
print(f"无折扣平均订单金额: ${avg_order_without_discount:.2f}")

## 分析结论

1. **畅销品类**: Electronics 和 Clothing 是主要收入来源
2. **高价值用户**: 前25%用户贡献了大部分销售额
3. **支付偏好**: Credit Card 是最常用的支付方式
4. **物流时效**: Premium 用户配送更快（差异化服务有效）
5. **折扣效果**: 有折扣时平均订单金额更高

## 建议

- 针对高价值用户提供专属优惠
- 加大畅销品类的促销力度
- 优化Regular用户的配送时效